# 🚌 ECO-SYNC: Master Colab Pipeline\n\nRun this entire notebook from top to bottom on a T4 GPU.

In [ ]:
!pip install gymnasium stable-baselines3 osmnx groq python-dotenv kaggle

In [ ]:
# Load Secrets securely from Google Colab
import os
try:
    from google.colab import userdata
    # Groq API Key
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ GROQ_API_KEY loaded successfully.")
    
    # Kaggle API Keys
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("✅ Kaggle credentials loaded successfully.")
    
    # Example: Download a Kaggle dataset 
    # !kaggle datasets download -d <your-dataset-name> --unzip -p ./data/raw
except Exception as e:
    print(f"⚠️ Warning: Could not load secrets from Colab. Ensure they are set in the sidebar. Error: {e}")


## Phase 0: The Bus Bunching Phenomenon (Visualization)\n\nBefore diving into the data, let us visualize why bus bunching occurs. This polar plot simulates 4 buses. If one gets slightly delayed, it encounters more passengers, causing further delays, while the bus behind it catches up.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# --- Simulation Parameters ---
NUM_BUSES = 4
ROUTE_LENGTH = 360  # Represented as degrees in a circle
NOMINAL_SPEED = 2.0 # Base speed
DELAY_FACTOR = 0.05 # How much speed drops per unit of gap (simulating passenger boarding)

# Initial setup: perfectly spaced buses
positions = np.linspace(0, ROUTE_LENGTH, NUM_BUSES, endpoint=False)
colors = ['green', 'blue', 'orange', 'red']

# Introduce a slight initial delay to Bus 0 to trigger the bunching cascade
positions[0] -= 10 

# --- Setup the Plot ---
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={'projection': 'polar'})
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1) # Clockwise
ax.set_rticks([]) # Hide radial ticks
ax.set_yticklabels([])
ax.set_title("Bus Bunching Phenomenon", va='bottom')

# Draw the route
route_circle = plt.Circle((0, 0), 1, transform=ax.transData._b, color='gray', fill=False, linewidth=2, alpha=0.5)
ax.add_patch(route_circle)

# Initialize bus markers
scat = ax.scatter(np.radians(positions), np.ones(NUM_BUSES), c=colors, s=150, zorder=3)

def update(frame):
    global positions
    
    # Calculate the gap to the bus ahead of it
    # We sort positions to easily find the next bus
    sorted_idx = np.argsort(positions)
    gaps = np.zeros(NUM_BUSES)
    
    for i in range(NUM_BUSES):
        current_bus = sorted_idx[i]
        next_bus = sorted_idx[(i + 1) % NUM_BUSES]
        
        # Calculate distance to next bus, wrapping around the circle
        gap = (positions[next_bus] - positions[current_bus]) % ROUTE_LENGTH
        gaps[current_bus] = gap

    # Update positions based on gap (Larger gap = more passengers = slower speed)
    # The ideal gap is ROUTE_LENGTH / NUM_BUSES (e.g., 90 degrees)
    ideal_gap = ROUTE_LENGTH / NUM_BUSES
    
    speeds = np.zeros(NUM_BUSES)
    for i in range(NUM_BUSES):
        # If gap is larger than ideal, speed decreases (boarding takes longer)
        # If gap is smaller, speed increases (fewer passengers to board)
        speed_modifier = 1.0 - ((gaps[i] - ideal_gap) * DELAY_FACTOR)
        
        # Ensure speed doesn't go negative and has a reasonable max
        speeds[i] = max(0.5, min(NOMINAL_SPEED * speed_modifier, NOMINAL_SPEED * 1.5))
        
        # Move the bus
        positions[i] = (positions[i] + speeds[i]) % ROUTE_LENGTH

    # Update scatter plot
    scat.set_offsets(np.c_[np.radians(positions), np.ones(NUM_BUSES)])
    return scat,

# --- Run the Animation ---
ani = animation.FuncAnimation(fig, update, frames=200, interval=50, blit=True)

# To display in Colab/Jupyter Notebook:
plt.close() # Prevent static plot from showing below the video
HTML(ani.to_html5_video())


## Phase A: Data Engineering & Graph Extraction

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy.stats import poisson
import osmnx as ox

print("Extracting OSM graph for Bengaluru (Full City)...")
try:
    graph = ox.graph_from_place('Bengaluru, Karnataka, India', network_type='drive')
    nodes, edges = ox.graph_to_gdfs(graph)
    print(f"Graph extracted with {len(nodes)} nodes and {len(edges)} edges.")
except Exception as e:
    print(f"Extraction failed: {e}")
    graph, nodes, edges = None, [], []

num_stops = 30
time_steps = 120
peak_arrivals = poisson.rvs(mu=8, size=(num_stops, 60))
offpeak_arrivals = poisson.rvs(mu=2, size=(num_stops, 60))
arrival_rates = np.hstack([peak_arrivals, offpeak_arrivals])

num_segments = num_stops - 1
delays_matrix = np.abs(np.random.normal(loc=45.0, scale=15.0, size=(num_segments, time_steps)))

economic_baselines = {
    'wait_cost_per_min': 1.67,
    'fuel_cost_per_min': 0.83,
    'ticket_price': 15.0
}

bengaluru_simulation_data = {
    'graph': graph,
    'graph_nodes_gdf': nodes,
    'graph_edges_gdf': edges,
    'arrival_rates': arrival_rates,
    'delays_matrix': delays_matrix,
    'economic_baselines': economic_baselines,
    'metadata': {
        'num_stops': num_stops,
        'time_steps': time_steps,
        'description': "Bengaluru Full Data for ECO-SYNC"
    }
}

os.makedirs("./data/processed", exist_ok=True)
output_path = "./data/processed/bengaluru_simulation_data.pkl"
with open(output_path, 'wb') as f:
    pickle.dump(bengaluru_simulation_data, f)
print(f"Data successfully generated and saved to {output_path}")


## Phase B: Simulation Environment (Gymnasium)

In [ ]:
import os
import pickle
import numpy as np
import gymnasium as gym
from gymnasium import spaces

class BengaluruBusEnv(gym.Env):
    def __init__(self):
        super(BengaluruBusEnv, self).__init__()
        
        # Load synthetic data
        data_path = os.path.join(
            os.path.dirname(os.path.abspath(__file__)),
            "data", "processed", "bengaluru_simulation_data.pkl"
        )
        with open(data_path, "rb") as f:
            self.data = pickle.load(f)
            
        self.num_stops = self.data['metadata']['num_stops']
        self.time_steps = self.data['metadata']['time_steps']
        self.arrival_rates = self.data['arrival_rates']
        self.delays_matrix = self.data['delays_matrix']
        self.eco_baselines = self.data['economic_baselines']
        
        self.num_buses = 5
        self.capacity = 80
        
        # Action space: 0: PROCEED, 1: HOLD, 2: SKIP
        self.action_space = spaces.Discrete(3)
        
        # Observation space: [gap_ahead, gap_behind, queue_length]
        self.observation_space = spaces.Box(low=0, high=np.inf, shape=(3,), dtype=np.float32)
        
        self.reset()
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.bus_pos = {i: (i * (self.num_stops // self.num_buses)) % self.num_stops for i in range(self.num_buses)}
        self.bus_load = {i: 0 for i in range(self.num_buses)}
        
        self.stops = {i: {'queue_length': 0} for i in range(self.num_stops)}
        
        return self._get_obs(0), {}
        
    def _get_obs(self, bus_id):
        ahead_id = (bus_id - 1) % self.num_buses
        behind_id = (bus_id + 1) % self.num_buses
        
        gap_ahead = (self.bus_pos[ahead_id] - self.bus_pos[bus_id]) % self.num_stops
        gap_behind = (self.bus_pos[bus_id] - self.bus_pos[behind_id]) % self.num_stops
        queue_len = self.stops[self.bus_pos[bus_id]]['queue_length']
        
        return np.array([gap_ahead, gap_behind, queue_len], dtype=np.float32)
        
    def compute_reward(self, bus_id, action):
        passengers_waiting = self.stops[self.bus_pos[bus_id]]['queue_length']
        passengers_onboard = self.bus_load[bus_id]
        
        # Calculate delays from synthetic matrix
        time_idx = min(self.current_step, self.time_steps - 1)
        segment = min(self.bus_pos[bus_id], self.num_stops - 2)
        delay_seconds = self.delays_matrix[segment, time_idx]
        self.delay_minutes = delay_seconds / 60.0
        
        # Skipping a stop avoids acceleration/deceleration/door delays, reducing delay by 30%
        if action == 2:
            self.delay_minutes *= 0.7
            
        idle_minutes = 1.0 if action == 1 else 0.0
        self.idle_or_delay_minutes = self.delay_minutes + idle_minutes
        
        # Micro: wait cost (Value of Time scaled 10x for proper economic penalty)
        wait_cost = passengers_waiting * 1.67 * self.delay_minutes * 10.0
        if action == 2:
            # Passenger inconvenience penalty for being passed by
            wait_cost += passengers_waiting * 15.0
            
        # Macro: BMTC operational fuel cost (Speed 15km/h = â‚¹0.83/min)
        fuel_cost = 0.83 * self.idle_or_delay_minutes
        
        # Revenue: ticket price â‚¹15 x boarded passengers (0 if skipping stop)
        if action == 2:
            boarded = 0
        else:
            boarded = min(passengers_waiting, self.capacity - passengers_onboard)
        revenue = 15 * boarded
        
        return revenue, wait_cost, fuel_cost

    def step(self, action):
        bus_id = self.current_step % self.num_buses
        stop = self.bus_pos[bus_id]
        
        time_idx = min(self.current_step, self.time_steps - 1)
        
        # Continuous passenger arrivals at all stops
        for s in range(self.num_stops):
            self.stops[s]['queue_length'] += self.arrival_rates[s, time_idx] / self.num_buses
            
        revenue, wait_cost, fuel_cost = self.compute_reward(bus_id, action)
        reward = revenue - wait_cost - fuel_cost
        
        # Check bunching to apply penalty
        ahead_id = (bus_id - 1) % self.num_buses
        gap_ahead = (self.bus_pos[ahead_id] - self.bus_pos[bus_id]) % self.num_stops
        if gap_ahead <= 1.0:
            reward -= 250.0  # Dynamic bunching penalty
            
        if action == 2: # SKIP
            # No boarding or alighting at this stop
            boarded = 0
            alighting = 0
        else:
            passengers_waiting = self.stops[stop]['queue_length']
            boarded = min(passengers_waiting, self.capacity - self.bus_load[bus_id])
            self.stops[stop]['queue_length'] -= boarded
            self.bus_load[bus_id] += boarded
            
            alighting = int(self.bus_load[bus_id] * 0.1)
            self.bus_load[bus_id] -= alighting
        
        if action == 0: # PROCEED
            self.bus_pos[bus_id] = (self.bus_pos[bus_id] + 1) % self.num_stops
        elif action == 1: # HOLD
            pass
        elif action == 2: # SKIP
            # Moves to the next stop but without boarding/alighting at the current one
            self.bus_pos[bus_id] = (self.bus_pos[bus_id] + 1) % self.num_stops
            
        self.current_step += 1
        done = self.current_step >= self.time_steps * self.num_buses
        
        next_bus_id = self.current_step % self.num_buses
        obs = self._get_obs(next_bus_id)
        
        info = {
            'revenue': float(revenue),
            'wait_cost': float(wait_cost),
            'fuel_cost': float(fuel_cost),
            'gap_ahead': float(obs[0]),
            'bunching': 1 if float(obs[0]) <= 1.0 else 0
        }
        
        return obs, float(reward), done, False, info

if __name__ == "__main__":
    env = BengaluruBusEnv()
    obs, info = env.reset()
    print("Environment Initialized Successfully.")
    print(f"Observation Space Shape: {env.observation_space.shape}")
    print(f"Action Space: {env.action_space}\n")

    print("Running 10 steps with random actions...")
    for i in range(10):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        print(f"Step {i+1}: Action={action}, Reward={reward:.2f}, Obs={obs}")

# Execution & Validation Cell
env = BengaluruBusEnv()
obs, info = env.reset()
print("Environment Initialized Successfully.")
print(f"Observation Space Shape: {env.observation_space.shape}")
print(f"Action Space: {env.action_space}\n")


## Phase C: Reinforcement Learning Training (PPO)

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

os.makedirs("./models", exist_ok=True)
env = BengaluruBusEnv()
vec_env = DummyVecEnv([lambda: env])

model = PPO("MlpPolicy", vec_env, verbose=1)
model.learn(total_timesteps=1000000)

model.save("./models/eco_sync_ppo")
print("Model saved successfully to ./models/eco_sync_ppo.zip")


## Phase D: LLM Reasoning Engine

In [ ]:
import os
import groq

def explain_action(bus_id, action, econ_data):
    action_str = {0: "PROCEED", 1: "HOLD", 2: "SKIP"}.get(action, "UNKNOWN")
    prompt = f"Bus {bus_id} chose to {action_str}. Economic calculation: Wait cost=\u20b9{econ_data['wait_cost']:.0f}, Fuel cost=\u20b9{econ_data['fuel_cost']:.0f}, Revenue=\u20b9{econ_data['revenue']:.0f}. Explain in 1 sentence why this was the economically optimal decision."
    
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise ValueError("GROQ_API_KEY environment variable is missing.")
    api_key = api_key.strip()
        
    client = groq.Groq(api_key=api_key)
    resp = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=80
    )
    return resp.choices[0].message.content.strip()

# Test
print("--- LLM Reasoning Test ---")
print(explain_action(2, 0, {'wait_cost': 45.0, 'fuel_cost': 22.0, 'revenue': 150.0}))


## Phase E: Cost-Benefit Analysis

In [ ]:
import os
import json
import numpy as np
import gymnasium as gym
from stable_baselines3 import PPO

# Import custom environment


def run_simulation(env, model=None, num_episodes=250):
    total_wait_costs = []
    total_fuel_costs = []
    total_revenues = []
    bunching_events_count = 0
    
    step_metrics = []
    stop_wait_times = {i: [] for i in range(env.num_stops)}
    
    for episode in range(num_episodes):
        obs, info = env.reset()
        done = False
        
        ep_wait_cost = 0
        ep_fuel_cost = 0
        ep_revenue = 0
        
        step_idx = 0
        while not done:
            if model is None:
                action = 0
            else:
                action, _states = model.predict(obs, deterministic=True)
                
            obs, reward, done, truncated, info = env.step(action)
            
            w = info.get('wait_cost', 0.0)
            f = info.get('fuel_cost', 0.0)
            r = info.get('revenue', 0.0)
            gap = info.get('gap_ahead', 0.0)
            
            ep_wait_cost += w
            ep_fuel_cost += f
            ep_revenue += r
            
            if gap <= 1:
                bunching_events_count += 1
                
            bus_id = env.current_step % env.num_buses
            stop_id = env.bus_pos[bus_id]
            stop_wait_times[stop_id].append(w)
            
            if episode == 0:
                step_metrics.append({'wait': w, 'fuel': f, 'revenue': r})
            else:
                if step_idx < len(step_metrics):
                    step_metrics[step_idx]['wait'] += w
                    step_metrics[step_idx]['fuel'] += f
                    step_metrics[step_idx]['revenue'] += r
                
            step_idx += 1
            
        total_wait_costs.append(ep_wait_cost)
        total_fuel_costs.append(ep_fuel_cost)
        total_revenues.append(ep_revenue)

    avg_wait_per_stop = []
    for stop_id, waits in stop_wait_times.items():
        if waits:
            avg_wait_per_stop.append(np.mean(waits))
        else:
            avg_wait_per_stop.append(0.0)
    equity_index = float(np.std(avg_wait_per_stop))

    time_series = []
    for i, m in enumerate(step_metrics):
        time_series.append({
            'time_step': i,
            'wait_cost': float(m['wait'] / num_episodes),
            'fuel_cost': float(m['fuel'] / num_episodes),
            'revenue': float(m['revenue'] / num_episodes)
        })

    return {
        'avg_wait_cost': float(np.mean(total_wait_costs)),
        'avg_fuel_cost': float(np.mean(total_fuel_costs)),
        'avg_revenue': float(np.mean(total_revenues)),
        'bunching_events': int(bunching_events_count),
        'equity_index': float(equity_index),
        'stop_performance': [float(x) for x in avg_wait_per_stop],
        'time_series': time_series
    }

def main():
    print("Initializing environment...")
    env = BengaluruBusEnv()
    
    print("Loading Agentic PPO model...")
    model_path = "./models/eco_sync_ppo"
    if os.path.exists(model_path + ".zip"):
        model = PPO.load(model_path)
    else:
        print("Model not found! Run Module 3 first.")
        return
        
    print("Running Group A (Static Baseline) - 250 episodes...")
    static_results = run_simulation(env, model=None, num_episodes=250)
    
    print("Running Group B (Agentic) - 250 episodes...")
    agentic_results = run_simulation(env, model=model, num_episodes=250)
    
    static_env_val = static_results['avg_revenue'] - (static_results['avg_wait_cost'] + static_results['avg_fuel_cost'])
    agentic_env_val = agentic_results['avg_revenue'] - (agentic_results['avg_wait_cost'] + agentic_results['avg_fuel_cost'])
    
    # Efficiency: higher ENV = better. Positive = agentic wins.
    efficiency_gain = ((agentic_env_val - static_env_val) / abs(static_env_val)) * 100 if static_env_val != 0 else 0
    
    results = {
        'summary': {
            'static': {
                'env': static_env_val,
                'wait_cost': static_results['avg_wait_cost'],
                'fuel_cost': static_results['avg_fuel_cost'],
                'revenue': static_results['avg_revenue'],
                'bunching_events': static_results['bunching_events'],
                'equity_index': static_results['equity_index']
            },
            'agentic': {
                'env': agentic_env_val,
                'wait_cost': agentic_results['avg_wait_cost'],
                'fuel_cost': agentic_results['avg_fuel_cost'],
                'revenue': agentic_results['avg_revenue'],
                'bunching_events': agentic_results['bunching_events'],
                'equity_index': agentic_results['equity_index']
            },
            'efficiency_gain_percent': efficiency_gain,
            'total_savings_rupees': (static_results['avg_wait_cost'] + static_results['avg_fuel_cost']) - (agentic_results['avg_wait_cost'] + agentic_results['avg_fuel_cost'])
        },
        'time_series': {
            'static': static_results['time_series'],
            'agentic': agentic_results['time_series']
        },
        'stop_performance': {
            'static': static_results['stop_performance'],
            'agentic': agentic_results['stop_performance']
        }
    }
    
    os.makedirs("results", exist_ok=True)
    with open("results/simulation_results.json", "w") as f:
        json.dump(results, f, indent=4)
        
    print("\n--- Simulation Complete ---")
    print(f"Static ENV:    Rs.{static_env_val:.2f}")
    print(f"Agentic ENV:   Rs.{agentic_env_val:.2f}")
    print(f"Efficiency Gain: {efficiency_gain:.2f}%")
    print("Results saved to results/simulation_results.json")

if __name__ == "__main__":
    main()
